# 💡 Notebook 5 — Insights & Recommendations

The final notebook translates the numbers into clear, actionable financial insights.
This is the most important skill for a data analyst — turning data into decisions.

**Insights covered:**
- Overall savings rate health
- Categories consistently over budget
- Side income growth trend
- Best and worst months
- Actionable recommendations

In [ ]:
import pandas as pd
import plotly.express as px

tx  = pd.read_csv('../data/cleaned.csv', parse_dates=['date'])
bud = pd.read_csv('../data/budgets.csv')
income   = tx[tx['type']=='Income'].copy()
expenses = tx[tx['type']=='Expense'].copy()
print('Data loaded ✅')

## 1 — Savings Rate Analysis

The 50/30/20 rule says: 50% needs, 30% wants, 20% savings. We measure against this.

In [ ]:
monthly_income   = income.groupby('month')['amount'].sum()
monthly_expenses = expenses.groupby('month')['amount'].sum()
savings_rate = ((monthly_income - monthly_expenses) / monthly_income * 100).round(1)
avg_rate = savings_rate.mean()

month_names = {1:'January', 2:'February', 3:'March'}
print('Savings Rate by Month:')
for m, rate in savings_rate.items():
    status = '✅' if rate >= 20 else ('⚠️ ' if rate >= 10 else '🔴')
    print(f'  {status} {month_names[m]}: {rate}%')
print(f'\nAverage savings rate: {avg_rate:.1f}%')
print(f'Target (20/20 rule): 20%')
if avg_rate >= 20:
    print('✅ Savings rate is healthy!')
else:
    shortfall = 20 - avg_rate
    print(f'⚠️  {shortfall:.1f}% below target. Need to reduce discretionary spend.')

## 2 — Category-Level Insights

In [ ]:
cat_monthly = expenses.groupby(['month','category'])['amount'].sum().reset_index().rename(columns={'amount':'actual'})
cat_monthly = cat_monthly.merge(bud, on='category', how='left')
cat_monthly['variance'] = cat_monthly['actual'] - cat_monthly['monthly_budget']

over_months = cat_monthly[cat_monthly['variance'] > 0].groupby('category').agg(
    months_over=('variance','count'),
    total_over=('variance','sum'),
    avg_over=('variance','mean')
).reset_index().sort_values('total_over', ascending=False)

print('Categories that went over budget:')
for _, row in over_months.iterrows():
    print(f'\n  📌 {row["category"]}')
    print(f'     Over budget in {int(row["months_over"])}/3 months')
    print(f'     Total overspend  : R {row["total_over"]:,.0f}')
    print(f'     Average per month: R {row["avg_over"]:,.0f}')

## 3 — Side Income Analysis

In [ ]:
side = income[income['category']=='Side Income'].groupby('month')['amount'].sum()
salary = income[income['category']=='Salary'].groupby('month')['amount'].sum()

print('Month-by-Month Income:')
for m in [1,2,3]:
    s  = salary.get(m, 0)
    si = side.get(m, 0)
    total = s + si
    side_pct = si / total * 100 if total > 0 else 0
    print(f'  {month_names[m]}: Salary R {s:,.0f}  |  Side Income R {si:,.0f}  ({side_pct:.1f}% of total)')

total_side   = side.sum()
total_income = income['amount'].sum()
print(f'\nSide income as % of 3-month total: {total_side/total_income*100:.1f}%')
print(f'Total freelance earned: R {total_side:,.0f}')

## 4 — Recommendations

Actionable steps based on the analysis. This is what makes a data analyst valuable.

In [ ]:
cat_totals = expenses.groupby('category')['amount'].sum()
food_total = cat_totals.get('Food', 0)
ent_total  = cat_totals.get('Entertainment', 0)
cloth_total= cat_totals.get('Clothing', 0)
savings_total = cat_totals.get('Savings', 0)
total_exp = expenses['amount'].sum()

print('='*60)
print('FINANCIAL RECOMMENDATIONS — Q1 2024')
print('='*60)
print()
print('1. FOOD SPENDING')
print(f'   You spent R {food_total:,.0f} on food over 3 months')
print(f'   (R {food_total/3:,.0f}/month vs budget R 6,000/month)')
print(f'   Tip: Meal prepping on Sundays can reduce takeaway spend.')
print()
print('2. CLOTHING')
print(f'   February clothing spend of R 2,800 was R 1,800 over budget.')
print(f'   Tip: Apply a 48-hour rule before non-essential clothing purchases.')
print()
print('3. SAVINGS CONSISTENCY')
print(f'   Savings dropped from R 2,000 to R 1,500 in Feb, recovered to R 3,500 in Mar.')
print(f'   Tip: Set up an automatic debit order on salary day so savings happen first.')
print()
print('4. SIDE INCOME')
print(f'   R {total_side:,.0f} earned from freelance over 3 months ({total_side/3:,.0f}/month avg).')
print(f'   Tip: Allocate 100% of side income to savings or debt — not lifestyle expenses.')
print()
print('5. ENTERTAINMENT')
print(f'   Over budget in 2 of 3 months. Set a strict monthly cash envelope.')
print()
print('='*60)

## 5 — Final Summary Dashboard

In [ ]:
total_income   = income['amount'].sum()
total_expenses = expenses['amount'].sum()
net_saved      = total_income - total_expenses
avg_savings    = savings_rate.mean()
total_side     = income[income['category']=='Side Income']['amount'].sum()

print('='*60)
print('3-MONTH FINANCIAL SUMMARY — JAN TO MAR 2024')
print('='*60)
print(f'Total Income         : R {total_income:,.2f}')
print(f'  of which Salary    : R {income[income["category"]=="Salary"]["amount"].sum():,.2f}')
print(f'  of which Freelance : R {total_side:,.2f}')
print(f'Total Expenses       : R {total_expenses:,.2f}')
print(f'Net Saved            : R {net_saved:,.2f}')
print(f'Average Savings Rate : {avg_savings:.1f}%')
print(f'Transactions logged  : {len(tx)}')
print('='*60)